# 07b · Gold — Matriculados (Hecho)

**Objetivo:** construir **solo el hecho de Matriculados** (`fact_matriculados.parquet`) a partir de `matriculados_clean_v2.parquet` y las **dimensiones ya guardadas** en `data/Gold/` por el notebook **07a**.

**Técnica:** pipeline **lazy + streaming** con `sink_parquet`, leyendo **solo las columnas necesarias** del scan y con `set_streaming_chunk_size(16 MB)` para mantener el pico de memoria por debajo de ~5 GB.

**Nota:** notebook **independiente** de 07a; solo requiere las `dim_*.parquet` en disco.

In [1]:
# Configuración: límite de hilos ANTES de importar Polars (12 núcleos)
import os
os.environ['POLARS_MAX_THREADS'] = '12'

import gc
from pathlib import Path

import polars as pl

pl.Config.set_streaming_chunk_size(16 * 1024 * 1024)  # 16 MB por lote (pico de memoria controlado)

print('polars', pl.__version__)
print('hilos activos:', pl.thread_pool_size())


def rss_actual_gb():
    '''RSS actual del proceso en GB (Linux, /proc/self/statm).'''
    try:
        with open('/proc/self/statm', encoding='utf-8') as fh:
            paginas = int(fh.read().split()[1])
        return paginas * os.sysconf('SC_PAGE_SIZE') / (1024**3)
    except (OSError, ValueError, IndexError):
        return float('nan')


print(f'RSS inicial: {rss_actual_gb():.2f} GB')
print()

# Rutas del proyecto (misma detección automática que los notebooks 01-07a)
current_dir = Path.cwd()
if (current_dir / 'data').exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / 'data').exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError('No se encontró la carpeta data. Ejecuta desde la raíz o desde notebooks/.')

SILVER = PROJECT_ROOT / 'data' / 'Silver'
GOLD = PROJECT_ROOT / 'data' / 'Gold'
MAT_V2 = SILVER / 'matriculados_clean_v2.parquet'

assert MAT_V2.exists(), f'No existe {MAT_V2.name}. Ejecuta primero 04_limpieza_matriculados.ipynb.'
assert (GOLD / 'dim_universidad.parquet').exists(), 'No hay dimensiones en data/Gold/. Ejecuta primero 07a_gold_ingresantes.ipynb.'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('Silver:', SILVER)
print('Gold:', GOLD)

polars 1.44.1
hilos activos: 12
RSS inicial: 0.08 GB

PROJECT_ROOT: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP
Silver: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Silver
Gold: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Gold


---
## SECCIÓN 1 · Recarga de dimensiones desde Gold

Las 5 dimensiones ya fueron construidas por `07a` y están en `data/Gold/dim_*.parquet`. Son tablas **pequeñas** (decenas/centenas de filas) → se cargan con `pl.read_parquet` (en RAM), no con `scan_parquet`.

In [2]:
# Recarga de dimensiones desde data/Gold (tablas pequeñas, en RAM)
dim_univ = pl.read_parquet(GOLD / 'dim_universidad.parquet')
dim_prog = pl.read_parquet(GOLD / 'dim_programa.parquet')
dim_periodo = pl.read_parquet(GOLD / 'dim_periodo.parquet')
dim_ubicacion = pl.read_parquet(GOLD / 'dim_ubicacion.parquet')
dim_local = pl.read_parquet(GOLD / 'dim_local.parquet')

for nombre, df in [
    ('DimUniversidad', dim_univ),
    ('DimPrograma', dim_prog),
    ('DimPeriodo', dim_periodo),
    ('DimUbicacion', dim_ubicacion),
    ('DimLocal', dim_local),
]:
    print(f'{nombre}: {df.height:,} filas · {len(df.columns)} cols')
print(f'RSS tras cargar dimensiones: {rss_actual_gb():.2f} GB')

DimUniversidad: 177 filas · 7 cols
DimPrograma: 537 filas · 8 cols
DimPeriodo: 18 filas · 5 cols
DimUbicacion: 98 filas · 4 cols
DimLocal: 65 filas · 7 cols
RSS tras cargar dimensiones: 0.10 GB


---
## SECCIÓN 2 · Construcción de `fact_matriculados.parquet` (streaming)

Pipeline lazy sobre `matriculados_clean_v2.parquet`:
1. **Proyección de columnas** en el scan (solo las necesarias): claves naturales + atributos degenerados.
2. Extracción de `ANIO` y `SEMESTRE` desde `PERIODO_ESTANDARIZADO` (`str.slice`).
3. **Joins** contra las dimensiones para obtener los `SK_*` (claves → FKs).
4. Selección final: `FK_Universidad, FK_Programa, FK_Periodo, FK_Ubicacion, FK_Local, GUID_PERSONA, SEXO, EDAD, NACIONALIDAD, Region_Sur`.
5. Guardado con `sink_parquet` (streaming); fallback `collect(engine='streaming').write_parquet` si fallara.

In [3]:
# --- FactMatriculados: claves → FK (streaming, sin OOM) ---
fact_mat = (
    pl.scan_parquet(MAT_V2)
    .select([
        'CODIGO_INEI', 'CODIGO_SIU_PROGRAMA', 'CODIGO_LOCAL',
        'DEPARTAMENTO_LOCAL', 'PROVINCIA_LOCAL', 'PERIODO_ESTANDARIZADO',
        'GUID_PERSONA', 'SEXO', 'EDAD', 'NACIONALIDAD', 'Region_Sur',
    ])
    .with_columns([
        pl.col('PERIODO_ESTANDARIZADO').str.slice(0, 4).cast(pl.Int64).alias('ANIO'),
        pl.col('PERIODO_ESTANDARIZADO').str.slice(5, 1).cast(pl.Int64).alias('SEMESTRE'),
    ])
    .join(dim_univ.select(['CODIGO_INEI', 'SK_Universidad']).lazy(), on='CODIGO_INEI', how='left')
    .join(dim_prog.select(['CODIGO_SIU_PROGRAMA', 'SK_Programa']).lazy(), on='CODIGO_SIU_PROGRAMA', how='left')
    .join(dim_periodo.select(['ANIO', 'SEMESTRE', 'SK_Periodo']).lazy(), on=['ANIO', 'SEMESTRE'], how='left')
    .join(
        dim_ubicacion.select(['DEPARTAMENTO', 'PROVINCIA', 'SK_Ubicacion']).lazy(),
        left_on=['DEPARTAMENTO_LOCAL', 'PROVINCIA_LOCAL'],
        right_on=['DEPARTAMENTO', 'PROVINCIA'],
        how='left',
    )
    .join(dim_local.select(['CODIGO_LOCAL', 'SK_Local']).lazy(), on='CODIGO_LOCAL', how='left')
    .select([
        'SK_Universidad', 'SK_Programa', 'SK_Periodo', 'SK_Ubicacion', 'SK_Local',
        'GUID_PERSONA', 'SEXO', 'EDAD', 'NACIONALIDAD', 'Region_Sur',
    ])
    .rename({
        'SK_Universidad': 'FK_Universidad',
        'SK_Programa': 'FK_Programa',
        'SK_Periodo': 'FK_Periodo',
        'SK_Ubicacion': 'FK_Ubicacion',
        'SK_Local': 'FK_Local',
    })
)

GOLD_FACT_MAT = GOLD / 'fact_matriculados.parquet'
GOLD_FACT_MAT.unlink(missing_ok=True)
try:
    fact_mat.sink_parquet(GOLD_FACT_MAT)
    print('Guardado con sink_parquet (streaming).')
except Exception as e:
    print(f'sink_parquet falló ({type(e).__name__}); usando collect(engine="streaming").write_parquet')
    fact_mat.collect(engine='streaming').write_parquet(GOLD_FACT_MAT)
print(f'→ {GOLD_FACT_MAT.name}')
print(f'Tamaño en disco: {GOLD_FACT_MAT.stat().st_size / 1e6:.1f} MB')
print(f'RSS tras guardar: {rss_actual_gb():.2f} GB')

Guardado con sink_parquet (streaming).
→ fact_matriculados.parquet
Tamaño en disco: 373.7 MB
RSS tras guardar: 0.42 GB


---
## SECCIÓN 3 · Validaciones

Usando `scan_parquet` + agregaciones / anti-joins **en streaming** sobre el archivo generado:
- **A** · Número de filas del hecho == número de filas de la fuente V2.
- **B** · 0 nulos en las FKs.
- **C** · 0 FKs huérfanas (cada FK existe en su dimensión).

In [4]:
print('VALIDACIONES — fact_matriculados')
print('=' * 78)

fact = pl.scan_parquet(GOLD_FACT_MAT)

# A) Conteo de filas vs fuente
src_n = pl.scan_parquet(MAT_V2).select(pl.len()).collect().item()
fact_n = fact.select(pl.len()).collect().item()
print(f'A) Filas: hecho={fact_n:,} · fuente={src_n:,} · coincide={fact_n == src_n}')
assert fact_n == src_n, 'El hecho no coincide con la fuente V2'
print()

# B) Nulos en FKs
print('B) Nulos por FK:')
fks = [c for c in fact.collect_schema().names() if c.startswith('FK_')]
nulos_total = 0
for fk in fks:
    n = fact.select(pl.col(fk).null_count()).collect().item()
    nulos_total += n
    print(f'  {fk}: {n:,}')
print(f'  Total nulos: {nulos_total:,} → {"OK" if nulos_total == 0 else "FALLO"}')
assert nulos_total == 0, 'Existen FKs nulas'
print()

# C) FKs huérfanas (anti-join en streaming)
print('C) FKs huérfanas:')
fk_map = {
    'FK_Universidad': ('dim_universidad.parquet', 'SK_Universidad'),
    'FK_Programa': ('dim_programa.parquet', 'SK_Programa'),
    'FK_Periodo': ('dim_periodo.parquet', 'SK_Periodo'),
    'FK_Ubicacion': ('dim_ubicacion.parquet', 'SK_Ubicacion'),
    'FK_Local': ('dim_local.parquet', 'SK_Local'),
}
huerfanos_total = 0
for fk in fks:
    dim_archivo, sk = fk_map[fk]
    dim = pl.scan_parquet(GOLD / dim_archivo)
    h = (
        fact.select(fk).unique()
        .join(dim.select(sk).unique(), left_on=fk, right_on=sk, how='anti')
        .select(pl.len())
        .collect()
        .item()
    )
    huerfanos_total += h
    print(f'  {fk}: {h:,}')
print(f'  Total huérfanos: {huerfanos_total:,} → {"OK" if huerfanos_total == 0 else "FALLO"}')
assert huerfanos_total == 0, 'Existen FKs huérfanas'
print('=' * 78)
print('Hecho de Matriculados validado.')

VALIDACIONES — fact_matriculados
A) Filas: hecho=17,368,424 · fuente=17,368,424 · coincide=True

B) Nulos por FK:
  FK_Universidad: 0
  FK_Programa: 0
  FK_Periodo: 0
  FK_Ubicacion: 0
  FK_Local: 0
  Total nulos: 0 → OK

C) FKs huérfanas:
  FK_Universidad: 0
  FK_Programa: 0


  FK_Periodo: 0
  FK_Ubicacion: 0
  FK_Local: 0
  Total huérfanos: 0 → OK
Hecho de Matriculados validado.


In [5]:
# Liberación de memoria
del fact_mat, dim_univ, dim_prog, dim_periodo, dim_ubicacion, dim_local
gc.collect()
print(f'RSS final: {rss_actual_gb():.2f} GB')
print('OK: Gold de Matriculados generado. Modelo estrella completo en data/Gold/.')

RSS final: 0.59 GB
OK: Gold de Matriculados generado. Modelo estrella completo en data/Gold/.
